# N7 — FinBERT Text Embeddings (M2)

Embeds Gemini 2.5 Flash 400-word business summaries using FinBERT
(ProsusAI/finbert) → 768-dimensional [CLS] token embeddings per firm-year.
Embeddings are L2-normalised before saving (required for cosine similarity).

**Input:** `business_summaries_{year}.csv` (N3)
**Output:** `text_embeddings.parquet` — schema: `tic | fyear | emb_0 ... emb_767`

**Runtime:** ~45 min GPU / ~3-4h CPU


In [1]:
# Cell 1 — install & imports
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'torch', 'transformers', 'sentencepiece', '-q'], check=True)

import sys; sys.path.insert(0, '..')
from config import *
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertModel

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Config loaded.")
print(f"  SUMMARIES_DIR : {SUMMARIES_DIR}")
print(f"  EMBEDDINGS    : {EMBEDDINGS}")
print(f"  MODEL         : {EMBEDDING_MODEL}")
print(f"  Device        : {device}")

Config loaded.
  SUMMARIES_DIR : /work/Repo/notebooks/../data/processed/Business_Summaries
  EMBEDDINGS    : /work/Repo/notebooks/../data/processed/text_embeddings.parquet
  MODEL         : ProsusAI/finbert
  Device        : cpu


In [2]:
# Cell 2 — declare I/O
INPUTS  = list(SUMMARIES_FILES.values())
OUTPUTS = [EMBEDDINGS]
# PURPOSE : FinBERT embeddings for Gemini summaries — 768-dim per firm-year
# RUNTIME : ~45 min GPU / ~3-4h CPU
# DEPENDS : business_summaries_{year}.csv (N3)


## 1. Load & Validate Summaries

In [3]:
# Cell 3 — load all summaries and validate
INVALID_FLAGS = {'ERROR', 'INSUFFICIENT_DATA', 'ERROR_EXTRACTING_TEXT'}

frames = []
for yr in YEARS:
    path = SUMMARIES_FILES[yr]
    if not path.exists():
        print(f"  {yr}: FILE NOT FOUND — {path}")
        continue
    df_yr = pd.read_csv(path, usecols=['tic', 'fyear', SUMMARY_COL])
    df_yr = df_yr[
        df_yr[SUMMARY_COL].notna() &
        ~df_yr[SUMMARY_COL].isin(INVALID_FLAGS) &
        (df_yr[SUMMARY_COL].str.len() > 50)
    ].copy()
    frames.append(df_yr)
    print(f"  {yr}: {len(df_yr):,} valid summaries")

df_text = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(df_text):,} firm-year summaries")
print(f"Unique firms: {df_text['tic'].nunique():,}")
print(f"Years: {sorted(df_text['fyear'].unique())}")


  2020: 2,446 valid summaries
  2021: 2,781 valid summaries
  2022: 2,717 valid summaries
  2023: 2,742 valid summaries
  2024: 2,873 valid summaries

Total: 13,559 firm-year summaries
Unique firms: 3,494
Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


## 2. Load FinBERT

In [4]:
# Cell 4 — load FinBERT tokenizer and model
print(f"Loading {EMBEDDING_MODEL}...")
tokenizer = BertTokenizer.from_pretrained(EMBEDDING_MODEL)
model     = BertModel.from_pretrained(EMBEDDING_MODEL, ignore_mismatched_sizes=True).to(device)
model.eval()
print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading ProsusAI/finbert...


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded. Parameters: 109,482,240


## 3. Embed — Year by Year with Checkpointing

In [5]:
# Cell 5 — embedding function
BATCH_SIZE = 64
def embed_texts(texts, tokenizer, model, device, batch_size=BATCH_SIZE):
    """
    Embed a list of texts using FinBERT mean pooling.
    Returns L2-normalised numpy array of shape (n, 768).
    """
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch,
            max_length=512,
            truncation=True,
            padding=True,
            return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            output = model(**encoded)
        hidden = output.last_hidden_state                        # (batch, seq, 768)
        mask   = encoded['attention_mask'].unsqueeze(-1).float()
        emb    = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)  # mean pool
        emb    = emb.cpu().numpy()
        norms  = np.linalg.norm(emb, axis=1, keepdims=True)
        emb    = emb / np.where(norms == 0, 1, norms)
        all_embeddings.append(emb)
    return np.vstack(all_embeddings)

print(f"embed_texts() defined.")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Max length : 512 tokens (truncation=True)")
print(f"  Pooling    : mean (non-padding tokens)")
print(f"  Normalise  : L2 per embedding")

embed_texts() defined.
  Batch size : 64
  Max length : 512 tokens (truncation=True)
  Pooling    : mean (non-padding tokens)
  Normalise  : L2 per embedding


In [6]:
# Cell 6 — embed year by year with checkpointing
EMBEDDINGS.parent.mkdir(parents=True, exist_ok=True)

yearly_parquets = []

for yr in YEARS:
    checkpoint = DATA_PROC / f"text_embeddings_{yr}.parquet"

    if checkpoint.exists():
        print(f"  {yr}: checkpoint found — loading")
        yearly_parquets.append(pd.read_parquet(checkpoint))
        continue

    df_yr = df_text[df_text['fyear'] == yr].reset_index(drop=True)
    if len(df_yr) == 0:
        print(f"  {yr}: no summaries — skipping")
        continue

    print(f"  {yr}: embedding {len(df_yr):,} summaries...")
    texts = df_yr[SUMMARY_COL].tolist()
    embs  = embed_texts(texts, tokenizer, model, device)

    # Build parquet: tic | fyear | emb_0 ... emb_767
    emb_cols = {f'emb_{i}': embs[:, i] for i in range(EMBEDDING_DIM)}
    df_emb = pd.DataFrame({'tic': df_yr['tic'].values,
                           'fyear': df_yr['fyear'].values,
                           **emb_cols})
    df_emb.to_parquet(checkpoint, index=False)
    yearly_parquets.append(df_emb)
    print(f"  {yr}: done → {checkpoint.name}")

# Concat all years
df_embeddings = pd.concat(yearly_parquets, ignore_index=True)
df_embeddings.to_parquet(EMBEDDINGS, index=False)

print(f"\nSaved: {EMBEDDINGS}")
print(f"Shape: {df_embeddings.shape[0]:,} rows × {df_embeddings.shape[1]} columns")
print(f"  tic + fyear + {EMBEDDING_DIM} embedding dimensions")


  2020: checkpoint found — loading
  2021: checkpoint found — loading
  2022: checkpoint found — loading


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

  2023: checkpoint found — loading
  2024: checkpoint found — loading

Saved: /work/Repo/notebooks/../data/processed/text_embeddings.parquet
Shape: 13,559 rows × 770 columns
  tic + fyear + 768 embedding dimensions


In [7]:
# Cell 7 — validate embeddings
emb_cols = [f'emb_{i}' for i in range(EMBEDDING_DIM)]

# Check L2 norms ≈ 1.0
norms = np.linalg.norm(df_embeddings[emb_cols].values, axis=1)
print(f"L2 norm check (should be ~1.0):")
print(f"  Mean: {norms.mean():.6f}")
print(f"  Std : {norms.std():.6f}")
print(f"  Min : {norms.min():.6f}")
print(f"  Max : {norms.max():.6f}")

print(f"\nNaN check: {df_embeddings[emb_cols].isna().sum().sum()} NaN values")
print(f"\nFirm-years per year:")
print(df_embeddings.groupby('fyear')['tic'].count().to_string())


L2 norm check (should be ~1.0):
  Mean: 1.000000
  Std : 0.000000
  Min : 0.999999
  Max : 1.000001

NaN check: 0 NaN values

Firm-years per year:
fyear
2020    2446
2021    2781
2022    2717
2023    2742
2024    2873


In [8]:
# Cell 8 — final summary
print("=" * 60)
print("N7 COMPLETE — FINBERT EMBEDDINGS SUMMARY")
print("=" * 60)
print(f"  Model     : {EMBEDDING_MODEL}")
print(f"  Dimension : {EMBEDDING_DIM}")
print(f"  mean (non-padding tokens)")
print(f"  Normalised: L2 per embedding")
print(f"  Firm-years: {len(df_embeddings):,}")
print(f"  Output    : {EMBEDDINGS}")
print()
print("  Next: N8 — kNN text peer lists (M2)")


N7 COMPLETE — FINBERT EMBEDDINGS SUMMARY
  Model     : ProsusAI/finbert
  Dimension : 768
  mean (non-padding tokens)
  Normalised: L2 per embedding
  Firm-years: 13,559
  Output    : /work/Repo/notebooks/../data/processed/text_embeddings.parquet

  Next: N8 — kNN text peer lists (M2)


In [9]:
import pandas as pd
from pathlib import Path

df_emb = pd.read_parquet('/work/Repo/data/processed/text_embeddings.parquet',
                          columns=['tic', 'fyear'])
SUMMARIES_DIR = Path('/work/Repo/data/processed/Business_Summaries')

print(f"Embeddings: {len(df_emb):,} firm-years")

# Check overlap with current valid summaries
summary_tickers = set()
for yr in [2020, 2021, 2022, 2023, 2024]:
    p = SUMMARIES_DIR / f'business_summaries_{yr}.csv'
    if p.exists():
        df_s = pd.read_csv(p)
        valid = df_s[df_s['business_description'].notna() &
                     (df_s['business_description'].str.len() > 50)]
        for _, row in valid.iterrows():
            summary_tickers.add((row['tic'], int(row['fyear'])))

emb_set = set(zip(df_emb['tic'], df_emb['fyear'].astype(int)))
in_both  = emb_set & summary_tickers
emb_only = emb_set - summary_tickers

print(f"Valid summaries: {len(summary_tickers):,}")
print(f"In both         : {len(in_both):,}")
print(f"Embeddings only : {len(emb_only):,} (will be excluded in N10 restriction)")

Embeddings: 13,559 firm-years
Valid summaries: 13,559
In both         : 13,559
Embeddings only : 0 (will be excluded in N10 restriction)


In [10]:
# Run in N7 or N10 wherever embeddings are loaded
import torch
from transformers import AutoTokenizer, AutoModel

# Take one known firm-year from your embeddings
sample_tic = embeddings_df['tic'].iloc[0]
sample_text = "Sample text to test"  # doesn't matter

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModel.from_pretrained("ProsusAI/finbert")

inputs = tokenizer(sample_text, return_tensors="pt", 
                   max_length=512, truncation=True, padding=True)
with torch.no_grad():
    outputs = model(**inputs)

cls_vec = outputs.last_hidden_state[:, 0, :]
mean_vec = outputs.last_hidden_state.mean(dim=1)

# Compare against your stored embedding for that firm
stored = embeddings_df[embeddings_df['tic']==sample_tic].iloc[0, 2:].values

import numpy as np
print(f"Cosine sim stored vs CLS:  {np.dot(stored, cls_vec.numpy().flatten()) / (np.linalg.norm(stored) * np.linalg.norm(cls_vec.numpy())):.6f}")
print(f"Cosine sim stored vs mean: {np.dot(stored, mean_vec.numpy().flatten()) / (np.linalg.norm(stored) * np.linalg.norm(mean_vec.numpy())):.6f}")

NameError: name 'embeddings_df' is not defined